# 05 · CoolWalk end-to-end demo

One notebook that runs the full pipeline for **Chennai · Delhi · Ahmedabad** and prints the hero-table numbers.

Run top to bottom; each cached city takes < 1 s. First-time OSM downloads take ~1 minute per new (lat, lon, radius) tuple.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from IPython.display import IFrame, display, Markdown

from src.config import CITIES
from src.data_loader import load_area
from src.trees import load_tree_cover_area
from src.shade import compute_edge_shade
from src.sun import get_sun_position
from src.routing import route_between
from src.metrics import summarize_route
from src.weather import fetch_conditions
from src.viz import render_routes

In [2]:
SCENES = [
    dict(name='Chennai T. Nagar 8 AM',   city='chennai',   lat=13.0418, lon=80.2341,
         when='2026-05-15 08:00', start=(13.0440, 80.2320), end=(13.0395, 80.2370)),
    dict(name='Delhi CP 9 AM',            city='delhi',     lat=28.6315, lon=77.2167,
         when='2026-05-15 09:00', start=(28.6345, 77.2140), end=(28.6285, 77.2200)),
    dict(name='Ahmedabad Manek Chowk 7 AM', city='ahmedabad', lat=23.0245, lon=72.5866,
         when='2026-05-15 07:00', start=(23.0275, 72.5845), end=(23.0210, 72.5895)),
]
ALPHA = 0.9
RADIUS_M = 800

In [3]:
rows = []
for scene in SCENES:
    city = CITIES[scene['city']]
    when = pd.Timestamp(scene['when'], tz=city.tz)

    graph, buildings = load_area(scene['lat'], scene['lon'], RADIUS_M)
    trees = load_tree_cover_area(scene['lat'], scene['lon'], RADIUS_M, dst_crs=buildings.crs)
    az, elev = get_sun_position(scene['lat'], scene['lon'], when)
    compute_edge_shade(graph, buildings, az, elev, tree_cover=trees)

    s = route_between(graph, *scene['start'], *scene['end'], alpha=0.0)
    c = route_between(graph, *scene['start'], *scene['end'], alpha=ALPHA)
    a = summarize_route(graph, s)
    b = summarize_route(graph, c)

    try:
        w = fetch_conditions(scene['lat'], scene['lon'], when)
        m = w.heat_multiplier()
        a, b = a.apply_weather(m), b.apply_weather(m)
        weather_s = w.summary()
    except Exception:
        weather_s = '—'

    rows.append({
        'Scene': scene['name'],
        'Sun (az/elev)': f'{az:.0f}° / {elev:.0f}°',
        'Weather': weather_s,
        'Shortest (m)': f'{a.distance_m:.0f}',
        'CoolWalk (m)': f'{b.distance_m:.0f}',
        'Shortest shaded': f'{100*a.shaded_fraction:.1f}%',
        'CoolWalk shaded': f'{100*b.shaded_fraction:.1f}%',
        'Sun saved (min)': f'{a.sun_exposure_min - b.sun_exposure_min:.1f}',
    })

pd.DataFrame(rows).set_index('Scene')

,Sun (az/elev),Weather,Shortest (m),CoolWalk (m),Shortest shaded,CoolWalk shaded,Sun saved (min)
Scene,,,,,,,
Chennai T. Nagar 8 AM,76° / 31°,27°C · UV 0.0 · 0 W/m² → ×1.42 felt load,1314,1506,32.9%,67.7%,4.9
Delhi CP 9 AM,91° / 44°,28°C · UV 0.0 · 0 W/m² → ×1.58 felt load,1628,1868,8.4%,46.4%,6.1
Ahmedabad Manek Chowk 7 AM,75° / 13°,28°C · UV 0.0 · 0 W/m² → ×1.62 felt load,1202,1697,11.0%,56.5%,4.1


In [4]:
# Render each scene as an inline folium map (shortest = red, CoolWalk = blue,
# grey = shadow layer at the given hour).
for scene in SCENES:
    city = CITIES[scene['city']]
    when = pd.Timestamp(scene['when'], tz=city.tz)
    graph, buildings = load_area(scene['lat'], scene['lon'], RADIUS_M)
    trees = load_tree_cover_area(scene['lat'], scene['lon'], RADIUS_M, dst_crs=buildings.crs)
    az, elev = get_sun_position(scene['lat'], scene['lon'], when)
    compute_edge_shade(graph, buildings, az, elev, tree_cover=trees)
    s = route_between(graph, *scene['start'], *scene['end'], alpha=0.0)
    c = route_between(graph, *scene['start'], *scene['end'], alpha=ALPHA)
    fmap = render_routes(graph, s, c, shadow_layer=graph.graph.get('shadow_layer'))
    display(Markdown(f'### {scene["name"]}'))
    display(fmap)

### Chennai T. Nagar 8 AM

### Delhi CP 9 AM

### Ahmedabad Manek Chowk 7 AM

Open the interactive Streamlit app for click-to-set routing:
```bash
streamlit run app/streamlit_app.py
```